# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and analyze the FAIR^2 dataset using the `mlcroissant` library, referencing all entities by their `@id` fields.

### Dataset Source
The dataset is described by a Croissant schema, available at:
`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant

## 1. Data Loading
Load dataset metadata and records using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json
from pprint import pprint

# Define the Croissant dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Print dataset metadata (access using attributes, not as a dict!)
print(f"{dataset.metadata.name}: {dataset.metadata.description}")

# Display basic metadata info
print("\nDataset identifier:", dataset.metadata.identifier)
print("Dataset version:", dataset.metadata.version)
print("Dataset license:", dataset.metadata.license)
print("Dataset authors:")
for author in getattr(dataset.metadata, 'author', []):
    print(" -", author.get('@id', author))

## 2. Data Overview
Inspect the available record sets and their fields using their `@id` fields.

In [ ]:
# List all available record sets and their field IDs
print("Available record sets (referenced by @id):")
record_sets = dataset.metadata.record_set  # This is a list of RecordSet objects, if available.

if not record_sets:
    print("Warning: No top-level record sets found directly in metadata. Attempting discovery from files...")
    # If record sets are not present, look into distributions and file objects in metadata
    if hasattr(dataset.metadata, 'distribution'):
        discovered_record_sets = []
        for dist in dataset.metadata.distribution:
            if hasattr(dist, 'record_set'):
                for rs in dist.record_set:
                    discovered_record_sets.append(rs)
        record_sets = discovered_record_sets

record_set_ids = []
for idx, rs in enumerate(record_sets):
    # rs should be a RecordSet object
    print(f"{idx+1}. @id: {rs['@id'] if isinstance(rs, dict) else getattr(rs,'@id','(no id)')}")
    rs_id = rs['@id'] if isinstance(rs, dict) else getattr(rs,'@id','(no id)')
    record_set_ids.append(rs_id)
    if hasattr(rs, 'field'):
        print("   Fields (@id):")
        for f in rs.field:
            if isinstance(f, dict) and '@id' in f:
                print(f"    - {f['@id']}")
            elif hasattr(f, '@id'):
                print(f"    - {f.@id}")
    else:
        print("   No fields found.")
if not record_set_ids:
    print("-- RecordSet IDs not found at root: you may need to obtain them from the DataFrame once loaded.")
    # Fallback: Try the standard croissant RecordSet ID (for demonstration)
    # Example default (you will replace this as needed below)
    record_set_ids = ['cr:RecordSet']

## 3. Data Extraction
Load data from each record set by its `@id` into a pandas DataFrame. Reference record sets and fields using their `@id` only.

In [ ]:
# Load data for each record set @id into pandas DataFrames
dfs = {}
for rs_id in record_set_ids:
    print(f"Loading records for record set: {rs_id}")
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dfs[rs_id] = df
    print(f"Columns for record set {rs_id}:\n", df.columns.tolist())
    print(df.head(), "\n")
# For further analysis, pick the first record set (if more, adjust as needed)
primary_record_set_id = record_set_ids[0] if record_set_ids else None
# Keep reference to its DataFrame
primary_df = dfs[primary_record_set_id]

## 4. Exploratory Data Analysis (EDA)
Let's process and analyze the main record set. We will:
- Select and filter records using a numeric field (by its `@id`)
- Normalize a numeric column
- Group by a categorical field if available

Please ensure you use the `@id` of the fields/columns shown above.

In [ ]:
# ------ Choose your field @ids below for demonstration ------ #
# You must select actual fields from your DataFrame/record_set as shown in prior output.
# For example purpose, let's pick two example fields by plausible names:
# E.g. 'cr:mw' for molecular_weight, 'cr:sample' for group/sample, etc.
numeric_field_id = None
group_field_id = None

# If you don't know the fields, use the previous print(df.columns) output to select them.
for col in primary_df.columns:
    if numeric_field_id is None and ('mw' in col.lower() or 'weight' in col.lower() or col.lower().startswith('cr:')):
        # Heuristically pick one numeric field
        if pd.api.types.is_numeric_dtype(primary_df[col]):
            numeric_field_id = col
    if group_field_id is None and ('sample' in col.lower() or 'group' in col.lower()):
        group_field_id = col

if numeric_field_id is None:
    print("No clear numeric field found, using the first numeric column (if any).")
    numerics = primary_df.select_dtypes(include='number').columns.tolist()
    if numerics:
        numeric_field_id = numerics[0]

if group_field_id is None:
    # Just use None if not found
    print("No group field found; grouping will be skipped.")

if numeric_field_id:
    print(f"Analyzing numeric field: {numeric_field_id}")
    threshold = primary_df[numeric_field_id].mean()  # Set threshold at mean for example
    filtered_df = primary_df[primary_df[numeric_field_id] > threshold]
    print(f"Filtered records from {primary_record_set_id} where {numeric_field_id} > {threshold:.2f}:")
    print(filtered_df.head())

    # Normalize the selected numeric field
    norm_col = numeric_field_id + '_normalized'
    filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, norm_col]].head())

    # Optionally group if group_field_id exists
    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"\nGrouped data by {group_field_id}, showing mean {numeric_field_id}:")
        print(grouped_df.head())
else:
    print("No numeric field available for analysis.")

## 5. Visualization
Let's visualize the distribution of the chosen numeric field and group means (if a grouping field is available).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id and numeric_field_id in primary_df.columns:
    plt.figure(figsize=(8,4))
    sns.histplot(primary_df[numeric_field_id].dropna(), bins=30, kde=True)
    plt.title(f'Distribution of {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.show()

    # If group_field_id exists, plot group means
    if group_field_id and group_field_id in primary_df.columns:
        group_means = primary_df.groupby(group_field_id)[numeric_field_id].mean().sort_values()
        plt.figure(figsize=(8,4))
        sns.barplot(x=group_means.index.astype(str), y=group_means.values)
        plt.title(f'Mean {numeric_field_id} by {group_field_id}')
        plt.xlabel(group_field_id)
        plt.ylabel(f'Mean {numeric_field_id}')
        plt.xticks(rotation=90)
        plt.show()

## 6. Conclusion
In this notebook, we loaded and explored the FAIR^2 dataset defining protein abundance, modifications, and sequences in human samples, referencing all data by their Croissant `@id` fields. We processed one main record set and analyzed a numeric column with filtering, normalization, and grouping. Visualizations illustrated the data structure for further analysis. You can extend the EDA and visualizations to other record sets and fields as needed.

_End of notebook._